# COMP90051 Project — Image Classification Robustness Under Degradations

**Research Question**: How does image classification performance degrade under occlusion, motion blur,
and low-light conditions, and which model class is most robust?

**Dataset**: Intel Image Classification (Kaggle)  
https://www.kaggle.com/datasets/puneet6060/intel-image-classification

**Models**:
1. Logistic Regression + HOG (simple)
2. RBF SVM + HOG (medium)
3. EfficientNet-B0 (complex) — Tan & Le, ICML 2019

**Notes**:
- Cross-validation, hyperparameter tuning, and all metrics are implemented **from scratch**
- `sklearn` / `torch` are used **only** for model algorithms
- Set `QUICK_RUN = True` for a fast test; `False` for full experiment

## 0. Imports & Config

In [1]:
import os
import random
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image

# skimage: HOG feature extraction (feature construction — not metrics/CV)
from skimage.feature import hog as skimage_hog
from skimage.color import rgb2gray

# scipy: motion blur convolution
from scipy.ndimage import convolve1d

# sklearn: model algorithms ONLY — NOT used for CV / metrics / hyperparameter tuning
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# PyTorch: EfficientNet-B0 (Tan & Le, ICML 2019)
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

warnings.filterwarnings('ignore')

# ─── Paths ────────────────────────────────────────────────────────────────────
# 下载 Intel Image Classification 后修改此路径
DATA_ROOT    = "/Users/lucazhu/Desktop/COMP90051_Project/intel_dataset"
RESULTS_PATH = "/Users/lucazhu/Desktop/COMP90051_Project/results.csv"

# ─── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ─── Dataset ──────────────────────────────────────────────────────────────────
CLASSES     = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
NUM_CLASSES = len(CLASSES)

HOG_IMG_SIZE = (100, 100)   # resize for HOG (original >= 100x100, so this is downsample)
EFF_IMG_SIZE = (224, 224)   # standard EfficientNet-B0 input

# ─── Cross-Validation ─────────────────────────────────────────────────────────
N_OUTER = 10   # outer k-fold  (requirement: k >= 10)
N_INNER = 3    # inner k-fold  (requirement: k >= 3)

# ─── Hyperparameter Grids ─────────────────────────────────────────────────────
# 每个模型至少 3 个候选值；期望中间值被最常选中（empirically verified later）
LR_C_GRID   = [0.01, 0.1, 1.0]       # Logistic Regression: L2 regularization C
SVM_C_GRID  = [0.1,  1.0, 10.0]      # RBF SVM: regularization C (gamma='scale')
EFF_LR_GRID = [1e-4, 1e-3, 1e-2]     # EfficientNet: learning rate

# ─── Degradation Conditions ───────────────────────────────────────────────────
DEGRADATIONS = {
    'clean':     [0],              # baseline
    'occlusion': [0.10, 0.20, 0.30],   # cutout fraction of image area
    'blur':      [5, 9, 13],           # motion blur kernel size (px)
    'low_light': [0.7, 0.5, 0.3],      # brightness scale factor
}

# ─── Compute Config ───────────────────────────────────────────────────────────
# QUICK_RUN=True: 快速测试用 (少数 epoch + 小子集)
# QUICK_RUN=False: 完整实验 (建议 GPU)
QUICK_RUN           = True
MAX_PER_CLASS       = 200 if QUICK_RUN else None   # None = 全部使用
EFF_EPOCHS          = 3   if QUICK_RUN else 10
EFF_BATCH           = 32
EFF_PRETRAINED      = False   # False = train from scratch; True = ImageNet transfer

DEVICE = (torch.device('mps')  if torch.backends.mps.is_available() else
          torch.device('cuda') if torch.cuda.is_available() else
          torch.device('cpu'))

print(f'Device : {DEVICE}')
print(f'Quick run: {QUICK_RUN}  |  max_per_class: {MAX_PER_CLASS}  |  EfficientNet epochs: {EFF_EPOCHS}')

ModuleNotFoundError: No module named 'skimage'

## 1. Dataset Verification & EDA

验证：
- 图像总数 ≥ 10,000
- 原始图像尺寸 ≥ 100×100（作业要求，不能 resize 来凑数）
- 类别分布（class imbalance）

In [ ]:
def find_split_root(data_root, split='train'):
    """
    自动检测 Intel 数据集的目录结构。
    Intel 数据集有两种常见结构：
      - seg_train/seg_train/<class>/  (Kaggle 默认)
      - seg_train/<class>/            (解压后可能)
    """
    candidates = (
        [f'seg_{split}/seg_{split}', f'seg_{split}', split, '']
        if split in ('train', 'test')
        else [split, '']
    )
    for c in candidates:
        root = os.path.join(data_root, c) if c else data_root
        if os.path.isdir(root) and any(
            os.path.isdir(os.path.join(root, cls)) for cls in CLASSES
        ):
            return root
    raise FileNotFoundError(
        f"Cannot find {split} split in {data_root}.\n"
        f"Make sure CLASSES={CLASSES} folders exist under one of: {candidates}"
    )


def load_paths_labels(split_root, max_per_class=None, seed=SEED):
    """从目录结构加载图像路径和标签（不读取像素）。"""
    paths, labels = [], []
    rng = random.Random(seed)
    for cls_idx, cls_name in enumerate(CLASSES):
        cls_dir = os.path.join(split_root, cls_name)
        if not os.path.isdir(cls_dir):
            print(f'  [WARN] missing class dir: {cls_dir}')
            continue
        imgs = sorted([
            os.path.join(cls_dir, f)
            for f in os.listdir(cls_dir)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        if max_per_class:
            rng.shuffle(imgs)
            imgs = imgs[:max_per_class]
        paths.extend(imgs)
        labels.extend([cls_idx] * len(imgs))
    return paths, np.array(labels, dtype=int)


# --- 验证图像尺寸 ---
def verify_sizes(paths, min_wh=100, sample_n=None):
    """检查原始图像是否满足 >= min_wh x min_wh 的要求。"""
    check = paths if sample_n is None else random.sample(paths, min(sample_n, len(paths)))
    sizes = []
    for p in check:
        with Image.open(p) as img:
            sizes.append(img.size)   # (W, H)
    ws = [s[0] for s in sizes]
    hs = [s[1] for s in sizes]
    valid = sum(1 for w, h in sizes if w >= min_wh and h >= min_wh)
    print(f'Checked {len(sizes)} images')
    print(f'  Width : min={min(ws)}, max={max(ws)}, median={int(np.median(ws))}')
    print(f'  Height: min={min(hs)}, max={max(hs)}, median={int(np.median(hs))}')
    print(f'  >= {min_wh}x{min_wh}: {valid}/{len(sizes)} ({100*valid/len(sizes):.1f}%)')
    return sizes


# 找到 train 根目录（test 集仅用于参考，不参与 CV）
train_root = find_split_root(DATA_ROOT, 'train')
print(f'Train root: {train_root}')

# 加载所有训练路径（不限 max_per_class，先统计完整数量）
all_paths, all_labels = load_paths_labels(train_root, max_per_class=None)
print(f'\nTotal training images: {len(all_paths)}')

# 图像尺寸验证（检查全部或随机抽样 2000 张）
print('\nSize verification (original images):')
_ = verify_sizes(all_paths, min_wh=100, sample_n=2000)

In [ ]:
# --- 类别分布可视化 ---
class_counts = np.bincount(all_labels, minlength=NUM_CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# 柱状图：每类数量
axes[0].bar(CLASSES, class_counts, color='steelblue')
axes[0].set_title('Class Distribution (Train)')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(class_counts):
    axes[0].text(i, v + 30, str(v), ha='center', fontsize=9)

# 饼图
axes[1].pie(class_counts, labels=CLASSES, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.savefig('/Users/lucazhu/Desktop/COMP90051_Project/fig_class_distribution.png', dpi=150)
plt.show()

print(f'Min samples/class: {class_counts.min()}  |  Max: {class_counts.max()}')
print(f'Imbalance ratio: {class_counts.max()/class_counts.min():.2f}x')

# 样本可视化：每类展示 3 张
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(7, 3*NUM_CLASSES//2))
for cls_idx, cls_name in enumerate(CLASSES):
    cls_paths = [p for p, l in zip(all_paths, all_labels) if l == cls_idx]
    for col, p in enumerate(cls_paths[:3]):
        ax = axes[cls_idx][col]
        ax.imshow(Image.open(p).resize((100, 100)))
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls_name, rotation=0, labelpad=40, va='center', fontsize=9)
plt.suptitle('Sample Images per Class', y=1.01)
plt.tight_layout()
plt.savefig('/Users/lucazhu/Desktop/COMP90051_Project/fig_samples.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Feature Construction: Degradation Functions

定义三种退化条件，每种 3 个强度等级：
| 类型 | 参数 | 级别 |
|------|------|------|
| Occlusion | cutout 面积比例 | 10% / 20% / 30% |
| Motion Blur | 水平 kernel 大小 (px) | 5 / 9 / 13 |
| Low-light | 亮度缩放因子 | 0.7 / 0.5 / 0.3 |

In [ ]:
def apply_occlusion(arr, fraction, seed=0):
    """
    Cutout 遮挡：随机在图像中心附近生成一个黑色方块，
    面积占图像总面积的 `fraction`。
    arr: np.uint8 (H, W, 3)
    """
    h, w = arr.shape[:2]
    side = int(np.sqrt(h * w * fraction))
    rng = np.random.default_rng(seed)
    x0 = rng.integers(0, max(1, w - side))
    y0 = rng.integers(0, max(1, h - side))
    out = arr.copy()
    out[y0:y0+side, x0:x0+side] = 0
    return out


def apply_motion_blur(arr, kernel_size):
    """
    水平运动模糊：使用 1D 均值 kernel 沿 x 轴卷积。
    固定水平方向（模拟行驶中的相机运动），避免随机方向引入不稳定方差。
    kernel_size: 奇数 (5, 9, 13)
    """
    kernel = np.ones(kernel_size, dtype=np.float32) / kernel_size
    out = arr.astype(np.float32)
    for c in range(3):
        out[:, :, c] = convolve1d(out[:, :, c], kernel, axis=1, mode='reflect')
    return np.clip(out, 0, 255).astype(np.uint8)


def apply_low_light(arr, scale, seed=0):
    """
    低光照模拟：缩放亮度 + 添加与暗度成比例的 Gaussian 噪声，
    更贴近真实夜间图像的统计特性（暗处噪声更明显）。
    scale: 亮度保留比例，越小越暗 (0.7 / 0.5 / 0.3)
    """
    out = arr.astype(np.float32) * scale
    sigma = 8.0 * (1.0 - scale)   # 越暗噪声越大
    rng = np.random.default_rng(seed)
    noise = rng.normal(0, sigma, out.shape)
    return np.clip(out + noise, 0, 255).astype(np.uint8)


def degrade_image(img_pil, deg_type, level, seed=0):
    """
    对 PIL 图像应用指定退化，返回 PIL 图像。
    deg_type: 'clean' | 'occlusion' | 'blur' | 'low_light'
    level: 对应强度参数（clean 时忽略）
    """
    if deg_type == 'clean':
        return img_pil
    arr = np.array(img_pil.convert('RGB'))
    if deg_type == 'occlusion':
        arr = apply_occlusion(arr, level, seed=seed)
    elif deg_type == 'blur':
        arr = apply_motion_blur(arr, level)
    elif deg_type == 'low_light':
        arr = apply_low_light(arr, level, seed=seed)
    return Image.fromarray(arr)


print('Degradation functions defined.')

In [ ]:
# --- 退化效果可视化（同一张图在各条件下的对比）---
sample_path = all_paths[0]
sample_img  = Image.open(sample_path).resize((150, 150))

rows = [('clean', 0), ('occlusion', 0.20), ('blur', 9), ('low_light', 0.4)]
level_cols = {
    'clean':     [0],
    'occlusion': [0.10, 0.20, 0.30],
    'blur':      [5, 9, 13],
    'low_light': [0.7, 0.5, 0.3],
}

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for row_idx, (deg_type, _) in enumerate(rows):
    levels = level_cols[deg_type]
    # 第一列：原图
    axes[row_idx][0].imshow(sample_img)
    axes[row_idx][0].set_title('Original', fontsize=9)
    axes[row_idx][0].axis('off')
    axes[row_idx][0].set_ylabel(deg_type, rotation=0, labelpad=50, va='center', fontsize=10)
    # 其余三列：3 个强度
    for col_idx, lvl in enumerate(levels):
        deg = degrade_image(sample_img, deg_type, lvl)
        axes[row_idx][col_idx+1].imshow(deg)
        axes[row_idx][col_idx+1].set_title(f'level={lvl}', fontsize=9)
        axes[row_idx][col_idx+1].axis('off')

plt.suptitle('Degradation Conditions (same image)', fontsize=12)
plt.tight_layout()
plt.savefig('/Users/lucazhu/Desktop/COMP90051_Project/fig_degradations.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Construction: HOG Extraction

HOG (Histogram of Oriented Gradients) 是用于 LR 和 SVM 的手工特征，属于 **advanced preprocessing**。  
对每张图（原图或退化后）：resize → 灰度 → 提取 HOG 向量。

为提高效率，预计算所有退化条件下的 HOG 特征矩阵并缓存到字典中。

In [ ]:
def extract_hog(img_pil, size=HOG_IMG_SIZE):
    """
    从 PIL 图像提取 HOG 特征向量。
    参数:
      orientations=9, pixels_per_cell=(8,8), cells_per_block=(2,2)
    对于 100x100 输入，输出维度 = 9 * ((100/8-1)^2 * 4) = 1764 维
    """
    img = img_pil.convert('RGB').resize(size, Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    gray = rgb2gray(arr)   # (H, W)
    feat = skimage_hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        feature_vector=True
    )
    return feat.astype(np.float32)


def build_hog_matrix(paths, deg_type='clean', level=0):
    """
    对一批图像路径：先施加退化，再提取 HOG，返回 (N, D) 特征矩阵。
    """
    feats = []
    for i, p in enumerate(paths):
        with Image.open(p) as img:
            img_deg = degrade_image(img, deg_type, level, seed=i)
            feats.append(extract_hog(img_deg))
    return np.array(feats, dtype=np.float32)


# --- 加载 working 子集（用于模型训练/CV）---
print(f'Loading paths with max_per_class={MAX_PER_CLASS} ...')
paths, labels = load_paths_labels(train_root, max_per_class=MAX_PER_CLASS)
print(f'Working set: {len(paths)} images, {NUM_CLASSES} classes')

# --- 预计算所有退化条件的 HOG 特征（缓存到字典）---
# hog_cache[(deg_type, level)] = np.array (N, D)
print('\nPrecomputing HOG features for all degradation conditions ...')
hog_cache = {}
for deg_type, levels in DEGRADATIONS.items():
    for level in levels:
        key = (deg_type, level)
        print(f'  {deg_type:12s} level={level}', end='\r')
        hog_cache[key] = build_hog_matrix(paths, deg_type=deg_type, level=level)

print(f'\nDone. HOG feature dim: {hog_cache[("clean", 0)].shape[1]}')
print(f'Total conditions cached: {len(hog_cache)}')

## 4. Metrics — Implemented from Scratch

> **作业要求**：不能使用 `sklearn.metrics` 等库计算实验结果，必须自行实现。

实现：
- `compute_accuracy`
- `compute_confusion_matrix`
- `compute_macro_f1`

In [ ]:
def compute_accuracy(y_true, y_pred):
    """Accuracy = 预测正确数 / 总样本数。"""
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sum(y_true == y_pred) / len(y_true))


def compute_confusion_matrix(y_true, y_pred, n_classes=NUM_CLASSES):
    """
    从头实现混淆矩阵。
    cm[i, j] = 真实标签为 i、预测为 j 的样本数。
    """
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    return cm


def compute_macro_f1(y_true, y_pred, n_classes=NUM_CLASSES):
    """
    从头实现 Macro-averaged F1。
    对每个类别计算 precision / recall / F1，再取平均。
    这种宏平均对类别不平衡更敏感，能反映模型在所有类上的整体表现。
    """
    cm = compute_confusion_matrix(y_true, y_pred, n_classes)
    f1_list = []
    for c in range(n_classes):
        tp = cm[c, c]
        fp = cm[:, c].sum() - tp   # 预测为 c 但实际不是 c
        fn = cm[c, :].sum() - tp   # 实际为 c 但预测不是 c
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
        f1_list.append(f1)
    return float(np.mean(f1_list))


def plot_confusion_matrix(cm, classes, title='Confusion Matrix', save_path=None):
    """可视化混淆矩阵（归一化为行概率）。"""
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=45, ha='right')
    ax.set_yticklabels(classes)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center',
                    color='white' if cm_norm[i, j] > 0.5 else 'black', fontsize=8)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


print('Metrics functions defined (from scratch — no sklearn.metrics).')

## 5. Cross-Validation — Implemented from Scratch

> **作业要求**：不能使用 `sklearn.model_selection` 中的 CV 工具，必须自行实现。

实现 Stratified K-Fold：按类别分桶后 round-robin 分配，保证每个 fold 的类别比例一致。

In [ ]:
def stratified_kfold(labels, k, seed=SEED):
    """
    Stratified k-fold split — from scratch.
    返回 k 个 (train_indices, test_indices) 元组的列表。
    算法：对每个类别单独 shuffle，然后 round-robin 分配到 k 个 fold。
    """
    labels = np.asarray(labels)
    n = len(labels)
    fold_ids = np.empty(n, dtype=int)
    rng = np.random.default_rng(seed)

    for cls in np.unique(labels):
        idx = np.where(labels == cls)[0]
        rng.shuffle(idx)
        for pos, i in enumerate(idx):
            fold_ids[i] = pos % k

    splits = []
    for fold in range(k):
        test_mask  = fold_ids == fold
        train_mask = ~test_mask
        splits.append((np.where(train_mask)[0], np.where(test_mask)[0]))
    return splits


def remap_inner_splits(outer_train_idx, labels, k_inner, seed=SEED):
    """
    在 outer_train_idx 子集上做 stratified k-fold，
    返回索引已映射回原始空间的 (train, val) 元组列表。
    """
    sub_labels = labels[outer_train_idx]
    sub_splits = stratified_kfold(sub_labels, k=k_inner, seed=seed)
    return [
        (outer_train_idx[tr], outer_train_idx[va])
        for tr, va in sub_splits
    ]


# 生成外层 10-fold splits（所有模型共用同一组 folds）
outer_splits = stratified_kfold(labels, k=N_OUTER, seed=SEED)

# 简单验证
fold_sizes = [len(te) for _, te in outer_splits]
print(f'Outer folds: {N_OUTER}')
print(f'Test fold sizes: min={min(fold_sizes)}, max={max(fold_sizes)}, total={sum(fold_sizes)}')
print(f'Total samples: {len(labels)} ✓' if sum(fold_sizes) == len(labels) else 'ERROR: fold sizes mismatch')

# 验证 stratification（每个 fold 中的类别比例应接近整体）
global_prop = np.bincount(labels) / len(labels)
max_deviation = 0.0
for _, te_idx in outer_splits:
    fold_prop = np.bincount(labels[te_idx], minlength=NUM_CLASSES) / len(te_idx)
    max_deviation = max(max_deviation, np.max(np.abs(fold_prop - global_prop)))
print(f'Max class proportion deviation across folds: {max_deviation:.4f}')

## 6. Model 1 — Logistic Regression (Simple)

- 特征：HOG
- 超参：正则化系数 `C` ∈ {0.01, 0.1, 1.0}
- Inner CV 选 `C`，Outer CV 评估各退化条件下的性能

In [ ]:
def nested_cv_hog_model(model_name, model_factory, hog_cache, labels,
                         outer_splits, param_grid, param_name,
                         n_inner=N_INNER):
    """
    通用 Nested CV 函数（用于 HOG-based 模型：LR 和 SVM）。

    流程：
      Outer fold (10x):
        Inner fold (3x) → 用 clean HOG 特征选最优超参 (macro-F1)
        → 用最优超参在 outer train 集重新训练
        → 在 outer test 集上评估所有退化条件

    返回：list of dicts，每条记录一个 (fold, model, deg_type, level) 的结果
    """
    records = []
    X_clean = hog_cache[('clean', 0)]   # 用 clean 特征做 inner CV

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_splits):
        print(f'  [{model_name}] outer fold {fold_idx+1}/{len(outer_splits)}', end='\r')

        # ── Inner CV：在 outer train 集上用 clean 特征选超参 ──────────────────
        inner_splits = remap_inner_splits(tr_idx, labels, k_inner=n_inner,
                                          seed=SEED + fold_idx)
        param_scores = defaultdict(list)

        for param_val in param_grid:
            for in_tr, in_va in inner_splits:
                # 特征标准化：fit 在 inner train，transform inner val
                scaler = StandardScaler()
                Xtr = scaler.fit_transform(X_clean[in_tr])
                Xva = scaler.transform(X_clean[in_va])
                ytr, yva = labels[in_tr], labels[in_va]

                mdl = model_factory(**{param_name: param_val})
                mdl.fit(Xtr, ytr)
                pred = mdl.predict(Xva)
                param_scores[param_val].append(compute_macro_f1(yva, pred))

        # 记录每个超参被选中次数（用于报告）
        mean_scores = {v: np.mean(s) for v, s in param_scores.items()}
        best_param  = max(mean_scores, key=mean_scores.get)

        # 统计每个 inner fold 中谁是最优（times_selected 分析）
        times_selected = defaultdict(int)
        for in_tr, in_va in inner_splits:
            fold_winner = max(
                param_grid,
                key=lambda v: compute_macro_f1(
                    labels[in_va],
                    model_factory(**{param_name: v}).fit(
                        StandardScaler().fit_transform(X_clean[in_tr]),
                        labels[in_tr]
                    ).predict(
                        StandardScaler().fit(X_clean[in_tr]).transform(X_clean[in_va])
                    )
                )
            )
            times_selected[fold_winner] += 1

        # ── 用最优超参在 outer train 全集上重训练 ────────────────────────────
        scaler_outer = StandardScaler()
        Xtr_outer = scaler_outer.fit_transform(X_clean[tr_idx])
        final_mdl = model_factory(**{param_name: best_param})
        final_mdl.fit(Xtr_outer, labels[tr_idx])

        # ── 在 outer test 上评估所有退化条件 ─────────────────────────────────
        for deg_type, deg_levels in DEGRADATIONS.items():
            for level in deg_levels:
                X_te_deg = hog_cache[(deg_type, level)][te_idx]
                X_te_deg_scaled = scaler_outer.transform(X_te_deg)
                pred = final_mdl.predict(X_te_deg_scaled)
                records.append({
                    'model':           model_name,
                    'fold':            fold_idx,
                    'deg_type':        deg_type,
                    'level':           level,
                    f'best_{param_name}': best_param,
                    'accuracy':        compute_accuracy(labels[te_idx], pred),
                    'macro_f1':        compute_macro_f1(labels[te_idx], pred),
                    'times_selected':  dict(times_selected),
                })

    print()
    return records


# --- 运行 LR Nested CV ---
print('Running Logistic Regression nested CV ...')
lr_records = nested_cv_hog_model(
    model_name    = 'LogisticRegression',
    model_factory = lambda C: LogisticRegression(
        C=C, max_iter=1000, random_state=SEED, solver='lbfgs', multi_class='multinomial'
    ),
    hog_cache    = hog_cache,
    labels       = labels,
    outer_splits = outer_splits,
    param_grid   = LR_C_GRID,
    param_name   = 'C',
)
print(f'LR done. Records: {len(lr_records)}')

In [ ]:
# LR 结果快速预览
lr_df = pd.DataFrame(lr_records)
lr_clean = lr_df[lr_df['deg_type'] == 'clean']
print('Logistic Regression — Clean performance (outer folds):')
print(f"  Accuracy : {lr_clean['accuracy'].mean():.4f} ± {lr_clean['accuracy'].std():.4f}")
print(f"  Macro-F1 : {lr_clean['macro_f1'].mean():.4f} ± {lr_clean['macro_f1'].std():.4f}")

# C 选中情况（验证中间值是否最常被选）
c_counts = defaultdict(int)
for r in lr_records:
    for k, v in r['times_selected'].items():
        c_counts[k] += v
print(f"\nC selection counts: {dict(c_counts)}")
print("（期望 C=0.1 被最常选中；如不满足，需调整候选值范围）")

## 7. Model 2 — RBF SVM (Medium)

- 特征：HOG
- 超参：`C` ∈ {0.1, 1.0, 10.0}，`gamma='scale'`（自动，= 1 / (n_features × X.var())）
- 使用与 LR 相同的 nested CV 框架，保证公平比较

In [ ]:
print('Running RBF SVM nested CV ...')
svm_records = nested_cv_hog_model(
    model_name    = 'RBF_SVM',
    model_factory = lambda C: SVC(
        C=C, kernel='rbf', gamma='scale', random_state=SEED, decision_function_shape='ovr'
    ),
    hog_cache    = hog_cache,
    labels       = labels,
    outer_splits = outer_splits,
    param_grid   = SVM_C_GRID,
    param_name   = 'C',
)
print(f'SVM done. Records: {len(svm_records)}')

svm_df = pd.DataFrame(svm_records)
svm_clean = svm_df[svm_df['deg_type'] == 'clean']
print('\nRBF SVM — Clean performance (outer folds):')
print(f"  Accuracy : {svm_clean['accuracy'].mean():.4f} ± {svm_clean['accuracy'].std():.4f}")
print(f"  Macro-F1 : {svm_clean['macro_f1'].mean():.4f} ± {svm_clean['macro_f1'].std():.4f}")

# C 选中情况
c_counts_svm = defaultdict(int)
for r in svm_records:
    for k, v in r['times_selected'].items():
        c_counts_svm[k] += v
print(f"\nC selection counts: {dict(c_counts_svm)}")

## 8. Model 3 — EfficientNet-B0 (Complex, ICML 2019)

> Tan, M. & Le, Q. V. (2019). *EfficientNet: Rethinking Model Scaling for Convolutional Neural Networks*. ICML.

- 特征：原始 RGB 图像（不使用 HOG），直接送入 CNN
- 超参：学习率 `lr` ∈ {1e-4, 1e-3, 1e-2}
- 退化在 DataLoader 中动态施加（transform 管道）
- 同样的 10-fold outer / 3-fold inner nested CV，保证公平比较

In [ ]:
class ImageFolderSubset(Dataset):
    """
    轻量 PyTorch Dataset：从预先收集的 (paths, labels) 中按 indices 子集加载图像。
    支持在加载时动态施加退化（degradation transform）。
    """
    def __init__(self, paths, labels, indices, transform,
                 deg_type='clean', level=0):
        self.paths     = [paths[i] for i in indices]
        self.labels    = labels[indices]
        self.transform = transform
        self.deg_type  = deg_type
        self.level     = level

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as img:
            img_pil = degrade_image(img, self.deg_type, self.level, seed=idx)
        tensor = self.transform(img_pil)
        return tensor, int(self.labels[idx])


# 标准 EfficientNet 预处理（ImageNet 均值/方差）
eff_train_transform = transforms.Compose([
    transforms.Resize(EFF_IMG_SIZE),
    transforms.RandomHorizontalFlip(),   # 训练时轻微数据增强
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])
eff_test_transform = transforms.Compose([
    transforms.Resize(EFF_IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])


def build_efficientnet(n_classes=NUM_CLASSES, pretrained=EFF_PRETRAINED):
    """
    构建 EfficientNet-B0。
    pretrained=False：从随机权重训练（控制实验）。
    pretrained=True ：ImageNet 迁移学习（可选）。
    修改最后分类层以适配 n_classes。
    """
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
    net = models.efficientnet_b0(weights=weights)
    in_features = net.classifier[1].in_features
    net.classifier[1] = nn.Linear(in_features, n_classes)
    return net.to(DEVICE)


def train_efficientnet(net, train_loader, lr, epochs=EFF_EPOCHS):
    """训练 EfficientNet，返回训练后的 net（原地修改）。"""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    net.train()
    for epoch in range(epochs):
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(net(imgs), lbls)
            loss.backward()
            optimizer.step()
    return net


def evaluate_efficientnet(net, loader):
    """推理并收集预测结果（不计算 loss）。"""
    net.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            preds = net(imgs.to(DEVICE)).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(lbls.numpy())
    return np.array(all_labels), np.array(all_preds)


print('EfficientNet utilities defined.')
print(f'Pretrained weights: {EFF_PRETRAINED}')

In [ ]:
def nested_cv_efficientnet(paths, labels, outer_splits,
                            lr_grid=EFF_LR_GRID, n_inner=N_INNER):
    """
    EfficientNet-B0 的 Nested CV。

    注意：每次 outer fold 需要重新训练模型，计算开销较大。
    建议在 GPU / Apple MPS 上运行，或使用 QUICK_RUN=True 减少 epoch。

    Inner CV 仅在 clean 图像上选学习率；
    Outer test 在所有退化条件上评估。
    """
    records = []

    for fold_idx, (tr_idx, te_idx) in enumerate(outer_splits):
        print(f'  [EfficientNet] outer fold {fold_idx+1}/{len(outer_splits)}')

        # ── Inner CV：选学习率（在 outer train、clean 条件下）──────────────
        inner_splits = remap_inner_splits(tr_idx, labels, k_inner=n_inner,
                                          seed=SEED + fold_idx)
        lr_scores = defaultdict(list)

        for lr_val in lr_grid:
            print(f'    inner CV for lr={lr_val}', end='')
            for i_fold, (in_tr, in_va) in enumerate(inner_splits):
                print(f' [{i_fold+1}]', end='', flush=True)
                tr_ds = ImageFolderSubset(paths, labels, in_tr, eff_train_transform)
                va_ds = ImageFolderSubset(paths, labels, in_va, eff_test_transform)
                tr_dl = DataLoader(tr_ds, batch_size=EFF_BATCH, shuffle=True,  num_workers=0)
                va_dl = DataLoader(va_ds, batch_size=EFF_BATCH, shuffle=False, num_workers=0)

                net = build_efficientnet()
                net = train_efficientnet(net, tr_dl, lr=lr_val)
                y_t, y_p = evaluate_efficientnet(net, va_dl)
                lr_scores[lr_val].append(compute_macro_f1(y_t, y_p))
                del net
                torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
            print()

        # 选最优 lr
        best_lr = max(lr_grid, key=lambda v: np.mean(lr_scores[v]))
        print(f'    Best lr for fold {fold_idx+1}: {best_lr}')

        # ── 用最优 lr 在 outer train 全集重训练 ──────────────────────────────
        tr_ds_full = ImageFolderSubset(paths, labels, tr_idx, eff_train_transform)
        tr_dl_full = DataLoader(tr_ds_full, batch_size=EFF_BATCH, shuffle=True, num_workers=0)
        net_final = build_efficientnet()
        net_final = train_efficientnet(net_final, tr_dl_full, lr=best_lr)

        # ── 在 outer test 上评估所有退化条件 ─────────────────────────────────
        for deg_type, deg_levels in DEGRADATIONS.items():
            for level in deg_levels:
                te_ds = ImageFolderSubset(
                    paths, labels, te_idx, eff_test_transform,
                    deg_type=deg_type, level=level
                )
                te_dl = DataLoader(te_ds, batch_size=EFF_BATCH, shuffle=False, num_workers=0)
                y_t, y_p = evaluate_efficientnet(net_final, te_dl)
                records.append({
                    'model':    'EfficientNet_B0',
                    'fold':     fold_idx,
                    'deg_type': deg_type,
                    'level':    level,
                    'best_lr':  best_lr,
                    'accuracy': compute_accuracy(y_t, y_p),
                    'macro_f1': compute_macro_f1(y_t, y_p),
                })

        del net_final
        torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

    return records


print('Running EfficientNet-B0 nested CV ...')
print(f'(This is the most expensive step — device={DEVICE}, epochs={EFF_EPOCHS})')
eff_records = nested_cv_efficientnet(paths, labels, outer_splits)
print(f'\nEfficientNet done. Records: {len(eff_records)}')

## 9. Results — Aggregation & Summary Table

In [ ]:
# 合并所有模型的结果
all_records = lr_records + svm_records + eff_records
df = pd.DataFrame(all_records)

# 保存原始 fold-level 结果到 CSV
df.to_csv(RESULTS_PATH, index=False)
print(f'Raw results saved to {RESULTS_PATH}')
print(f'Total records: {len(df)}')

# 聚合：mean ± std（跨 10 个 outer folds）
summary = (
    df.groupby(['model', 'deg_type', 'level'])
    .agg(
        acc_mean  = ('accuracy', 'mean'),
        acc_std   = ('accuracy', 'std'),
        f1_mean   = ('macro_f1', 'mean'),
        f1_std    = ('macro_f1', 'std'),
    )
    .reset_index()
)

# 打印 clean 条件下各模型汇总
clean_summary = summary[summary['deg_type'] == 'clean'].copy()
clean_summary['Accuracy (mean±std)'] = clean_summary.apply(
    lambda r: f"{r['acc_mean']:.4f} ± {r['acc_std']:.4f}", axis=1
)
clean_summary['Macro-F1 (mean±std)'] = clean_summary.apply(
    lambda r: f"{r['f1_mean']:.4f} ± {r['f1_std']:.4f}", axis=1
)
print('\n=== Clean Condition Performance (10-fold outer CV, mean ± std) ===')
print(clean_summary[['model', 'Accuracy (mean±std)', 'Macro-F1 (mean±std)']].to_string(index=False))

## 10. Visualization

### 10a. Degradation Curves
x = 退化强度，y = Macro-F1（每个模型一条线，带 error bars）  
这是回答 Research Question 的核心图表。

In [ ]:
MODEL_COLORS = {
    'LogisticRegression': 'steelblue',
    'RBF_SVM':            'darkorange',
    'EfficientNet_B0':    'forestgreen',
}
MODEL_LABELS = {
    'LogisticRegression': 'LR + HOG (simple)',
    'RBF_SVM':            'RBF SVM + HOG (medium)',
    'EfficientNet_B0':    'EfficientNet-B0 (complex)',
}

DEG_XLABELS = {
    'occlusion': ['10%', '20%', '30%'],
    'blur':      ['kernel=5', 'kernel=9', 'kernel=13'],
    'low_light': ['scale=0.7', 'scale=0.5', 'scale=0.3'],
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, (deg_type, levels) in zip(axes, [
    ('occlusion', [0.10, 0.20, 0.30]),
    ('blur',      [5, 9, 13]),
    ('low_light', [0.7, 0.5, 0.3]),
]):
    x = np.arange(len(levels))

    for model in ['LogisticRegression', 'RBF_SVM', 'EfficientNet_B0']:
        means, stds = [], []
        for level in levels:
            sub = df[(df['model'] == model) &
                     (df['deg_type'] == deg_type) &
                     (df['level'] == level)]['macro_f1']
            means.append(sub.mean())
            stds.append(sub.std())

        # Clean baseline（水平虚线）
        clean_f1 = df[(df['model'] == model) &
                      (df['deg_type'] == 'clean')]['macro_f1'].mean()

        color = MODEL_COLORS[model]
        ax.errorbar(x, means, yerr=stds,
                    marker='o', color=color,
                    label=MODEL_LABELS[model],
                    capsize=4, linewidth=2)
        ax.axhline(clean_f1, color=color, linestyle='--', alpha=0.4, linewidth=1)

    ax.set_xticks(x)
    ax.set_xticklabels(DEG_XLABELS[deg_type])
    ax.set_title(f'Degradation: {deg_type}', fontsize=12)
    ax.set_xlabel('Degradation Level', fontsize=10)
    ax.set_ylabel('Macro-F1', fontsize=10)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Degradation Curves (mean ± std, 10-fold outer CV)', fontsize=13)
plt.tight_layout()
plt.savefig('/Users/lucazhu/Desktop/COMP90051_Project/fig_degradation_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()

### 10b. Bar Chart — Clean vs Heavy Degradation

In [ ]:
# 对比：clean vs 最高强度退化
conditions = [
    ('clean',     0,    'Clean'),
    ('occlusion', 0.30, 'Occlusion\n30%'),
    ('blur',      13,   'Blur\nkernel=13'),
    ('low_light', 0.3,  'Low-light\nscale=0.3'),
]

models_ordered = ['LogisticRegression', 'RBF_SVM', 'EfficientNet_B0']
n_cond   = len(conditions)
n_models = len(models_ordered)
bar_w    = 0.22
x = np.arange(n_cond)

fig, ax = plt.subplots(figsize=(11, 5))

for m_idx, model in enumerate(models_ordered):
    means, stds = [], []
    for deg_type, level, _ in conditions:
        sub = df[(df['model'] == model) &
                 (df['deg_type'] == deg_type) &
                 (df['level'] == level)]['macro_f1']
        means.append(sub.mean())
        stds.append(sub.std())

    offset = (m_idx - 1) * bar_w
    bars = ax.bar(x + offset, means, bar_w, yerr=stds,
                  label=MODEL_LABELS[model],
                  color=MODEL_COLORS[model],
                  capsize=4, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([c[2] for c in conditions])
ax.set_ylabel('Macro-F1 (mean ± std)')
ax.set_title('Model Comparison: Clean vs Heavy Degradation')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/Users/lucazhu/Desktop/COMP90051_Project/fig_bar_chart.png',
            dpi=150, bbox_inches='tight')
plt.show()

### 10c. Confusion Matrix — Best Model on Clean Condition

In [ ]:
# 用最后一个 outer fold 的 clean 测试集重现最优模型预测（用于混淆矩阵可视化）
# 取 EfficientNet（预期最优）最后一个 outer fold
last_fold = N_OUTER - 1
_, te_idx = outer_splits[last_fold]

# 重建 SVM（作为示例；EfficientNet 需要 GPU 重训，此处用 SVM 演示混淆矩阵）
X_clean = hog_cache[('clean', 0)]
tr_idx_last, te_idx_last = outer_splits[last_fold]

scaler_cm = StandardScaler()
Xtr_cm = scaler_cm.fit_transform(X_clean[tr_idx_last])
Xte_cm = scaler_cm.transform(X_clean[te_idx_last])

# 使用 SVM 最常选的超参（从 summary 获取）
svm_cm = SVC(C=1.0, kernel='rbf', gamma='scale', random_state=SEED)
svm_cm.fit(Xtr_cm, labels[tr_idx_last])
pred_cm = svm_cm.predict(Xte_cm)

cm = compute_confusion_matrix(labels[te_idx_last], pred_cm)
plot_confusion_matrix(
    cm, CLASSES,
    title='Confusion Matrix — RBF SVM, Clean, Fold 10',
    save_path='/Users/lucazhu/Desktop/COMP90051_Project/fig_confusion_matrix.png'
)

### 10d. Hyperparameter Selection Summary

验证各模型超参的中间值是否被最常选中（作业 2f 要求）。

In [ ]:
# LR: C 选中次数汇总
lr_c_tally = defaultdict(int)
for r in lr_records:
    for v, cnt in r['times_selected'].items():
        lr_c_tally[v] += cnt

# SVM: C 选中次数汇总
svm_c_tally = defaultdict(int)
for r in svm_records:
    for v, cnt in r['times_selected'].items():
        svm_c_tally[v] += cnt

# EfficientNet: lr 选中次数汇总
eff_lr_tally = defaultdict(int)
for r in eff_records:
    eff_lr_tally[r.get('best_lr', None)] += 1

print('=== Hyperparameter Selection Counts ===')
print(f'\nLogistic Regression C grid = {LR_C_GRID}')
for v in LR_C_GRID:
    print(f'  C={v}: selected {lr_c_tally.get(v,0)} times')

print(f'\nRBF SVM C grid = {SVM_C_GRID}')
for v in SVM_C_GRID:
    print(f'  C={v}: selected {svm_c_tally.get(v,0)} times')

print(f'\nEfficientNet lr grid = {EFF_LR_GRID}')
for v in EFF_LR_GRID:
    print(f'  lr={v}: selected {eff_lr_tally.get(v,0)} times')

print('\n（若边界值被最多次选中，需要调整候选值范围后重跑实验）')

## 11. Full Results Table (for Report)

输出可直接复制进报告的汇总表格（mean ± std）。

In [ ]:
# 生成完整汇总表（所有模型 × 所有退化条件）
report_rows = []
for model in ['LogisticRegression', 'RBF_SVM', 'EfficientNet_B0']:
    for deg_type, levels in DEGRADATIONS.items():
        for level in levels:
            sub = df[(df['model'] == model) &
                     (df['deg_type'] == deg_type) &
                     (df['level'] == level)]
            if len(sub) == 0:
                continue
            report_rows.append({
                'Model':     MODEL_LABELS[model],
                'Condition': f'{deg_type}={level}',
                'Acc':       f"{sub['accuracy'].mean():.3f}±{sub['accuracy'].std():.3f}",
                'F1':        f"{sub['macro_f1'].mean():.3f}±{sub['macro_f1'].std():.3f}",
            })

report_df = pd.DataFrame(report_rows)
print(report_df.to_string(index=False))
report_df.to_csv('/Users/lucazhu/Desktop/COMP90051_Project/results_summary.csv', index=False)
print('\nSummary table saved to results_summary.csv')